# GRACE Gap Filling and Recharge Estimation

This notebook takes a groundwater storage anomaly (GWSa) time series exported from the
**GRACE Regional Analyst** web app and:

1. fills the months that have no GRACE data,
2. estimates annual recharge with the water table fluctuation (WTF) method, and
3. exports the filled series and the recharge table as CSV files.

The gap-filling and recharge steps follow Barbosa et al. (2022), *Evaluating Groundwater
Storage Change and Recharge Using GRACE Data: A Case Study of Aquifers in Niger, West
Africa*, Remote Sensing 14(7):1532, https://doi.org/10.3390/rs14071532.

**To run:** edit the settings in the first code cell, then choose *Runtime > Run all* in
Google Colab (or *Run All* in Jupyter). The notebook needs only pandas, numpy and
matplotlib, which Colab already has.

## 1. Setup and input

| Setting | Meaning |
|---|---|
| `CSV_FILE` | CSV exported from the web app. The default is a sample export for the Iullemeden-Irhazer Aquifer System (Global Aquifers set), downloaded automatically if it isn't already here. In Colab, any other name that isn't found opens a file upload dialog. |
| `VARIABLE` | Column to analyze. `"GWSa"` (groundwater) is the usual choice; `"TWSa"` also works. |
| `N_BREAKPOINTS` | Number of slope changes in the long-term trend: `"auto"` picks 0–3 by BIC, or give an integer 0–3. |
| `WATER_YEAR_START_MONTH` | First month of the water year (1–12), or `"auto"` to start at the month where storage is usually lowest. |
| `OVERRIDES` | Manual peak/trough picks for particular water years, e.g. `{2010: {"peak": "2010-09", "trough": "2010-05"}}`. Leave as `{}` at first, look at the plots in Section 4, and add entries only where an automatic pick looks wrong. |

In [ ]:
CSV_FILE = "sample_iullemeden.csv"
VARIABLE = "GWSa"
N_BREAKPOINTS = "auto"           # "auto" or 0, 1, 2, 3
WATER_YEAR_START_MONTH = "auto"  # "auto" or 1-12
OVERRIDES = {}                   # e.g. {2010: {"peak": "2010-09", "trough": "2010-05"}}

The next cell defines every function the notebook uses, so you can read the whole method in
one place. Run it; you don't need to edit it.

In [ ]:
import itertools
import sys
import urllib.request
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

COLORS = {
    "observed": "#1f3a5f",   # navy
    "blue": "#2b6cb0",
    "trend": "#2c7a7b",      # teal
    "filled": "#dd6b20",     # orange
    "recession": "#c53030",  # red
    "gray": "#718096",
}
ONE_MONTH = pd.DateOffset(months=1)
MIN_SEGMENT_MONTHS = 36  # shortest trend segment between two breakpoints
END_BUFFER_MONTHS = 48   # closest a breakpoint may be to either end of the record
MIN_BIC_DROP = 10        # BIC must fall by at least this much to accept more breakpoints
MIN_RECESSION_MONTHS = 4 # fewest months used to fit a recession line


def set_plot_style():
    """Plain white plots, sans-serif text, light horizontal gridlines only."""
    plt.rcParams.update({
        "figure.figsize": (9, 4),
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "font.family": "sans-serif",
        "font.sans-serif": ["Helvetica", "Arial", "Liberation Sans", "DejaVu Sans"],
        "font.size": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#4a5568",
        "axes.grid": True,
        "axes.grid.axis": "y",
        "grid.color": "#e2e8f0",
        "grid.linewidth": 0.8,
        "xtick.color": "#4a5568",
        "ytick.color": "#4a5568",
        "legend.frameon": False,
        "mathtext.default": "regular",  # subscripts like S_P in the regular font
        "svg.fonttype": "none",  # keep SVG text editable
    })


# ----------------------------------------------------------------------------
# Loading
# ----------------------------------------------------------------------------

def load_grace_csv(path, variable="GWSa"):
    """Read a GRACE Regional Analyst CSV into a DataFrame with one row per month.

    Months missing from the file, or present with a blank value, become NaN.
    The series is trimmed to the first and last month that have data.
    Columns: observed, upper, lower (upper/lower are value +/- 1 sigma).
    """
    raw = pd.read_csv(path)
    if variable not in raw.columns:
        raise ValueError(f"Column '{variable}' not found. Columns are: {list(raw.columns)}")

    # Snap every date to the first of its month so the index is regular.
    raw["Date"] = pd.to_datetime(raw["Date"]).dt.to_period("M").dt.to_timestamp()
    raw = raw.drop_duplicates("Date").set_index("Date").sort_index()

    has_data = raw.index[raw[variable].notna()]
    months = pd.date_range(has_data.min(), has_data.max(), freq="MS", name="Date")

    df = pd.DataFrame(index=months)
    df["observed"] = raw[variable].reindex(months).astype(float)
    for side in ("upper", "lower"):
        column = f"{variable}_{side}"
        if column in raw.columns:
            df[side] = raw[column].reindex(months).astype(float)
        else:
            df[side] = np.nan
    return df


def months_since(dates, start):
    """Whole months from `start` to each date (0 for the start month)."""
    dates = pd.DatetimeIndex(dates)
    return np.asarray((dates.year - start.year) * 12 + (dates.month - start.month))


def missing_runs(is_missing):
    """List (first_month, last_month) for each run of consecutive missing months."""
    runs = []
    start = None
    for date, missing in is_missing.items():
        if missing and start is None:
            start = date
        if not missing and start is not None:
            runs.append((start, date - ONE_MONTH))
            start = None
    if start is not None:
        runs.append((start, is_missing.index[-1]))
    return runs


# ----------------------------------------------------------------------------
# Gap filling: trend (piecewise linear) + seasonal (monthly means)
# ----------------------------------------------------------------------------

def design_matrix(t, calendar_month, breakpoints):
    """Regression columns for trend + seasonal.

    t is time in months. The trend part is t plus one hinge column max(t - b, 0) per
    breakpoint b, which keeps the trend continuous while letting the slope change at b.
    The seasonal part is one 0/1 column per calendar month. Together the 12 monthly
    columns play the role of the intercept, so there is no separate constant column.
    """
    trend_cols = [t] + [np.maximum(t - b, 0) for b in breakpoints]
    month_cols = [(calendar_month == m).astype(float) for m in range(1, 13)]
    return np.column_stack(trend_cols + month_cols).astype(float)


def fit_trend_seasonal(t, calendar_month, y, breakpoints):
    """Least-squares fit of trend + seasonal for fixed breakpoints. Returns a dict."""
    X = design_matrix(t, calendar_month, breakpoints)
    coef, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    rss = float(np.sum((y - X @ coef) ** 2))
    n = len(y)
    # Parameters: 1 slope + 1 slope change per breakpoint + 1 location per breakpoint
    # + 12 monthly levels.
    k = 1 + 2 * len(breakpoints) + 12
    bic = n * np.log(rss / n) + k * np.log(n)
    return {"breakpoints": list(breakpoints), "coef": coef, "rss": rss, "bic": bic, "n": n}


def breakpoints_valid(breakpoints, t_first, t_last, min_segment, end_buffer):
    """True if breakpoints are at least min_segment months apart and at least end_buffer
    months from either end of the record."""
    breakpoints = list(breakpoints)
    if breakpoints[0] - t_first < end_buffer or t_last - breakpoints[-1] < end_buffer:
        return False
    return all(b - a >= min_segment for a, b in zip(breakpoints[:-1], breakpoints[1:]))


def best_breakpoints(t, calendar_month, y, n_breakpoints, min_segment=MIN_SEGMENT_MONTHS,
                     end_buffer=END_BUFFER_MONTHS, step=3):
    """Find the n breakpoint months that give the smallest residual sum of squares.

    Grid search over candidate months spaced `step` months apart, then each breakpoint is
    nudged one month at a time until the fit stops improving. Returns None if the record
    is too short for this many segments.
    """
    t_first, t_last = int(t.min()), int(t.max())
    if n_breakpoints == 0:
        return fit_trend_seasonal(t, calendar_month, y, [])

    candidates = range(t_first + end_buffer, t_last - end_buffer + 1, step)
    best = None
    for combo in itertools.combinations(candidates, n_breakpoints):
        if not breakpoints_valid(combo, t_first, t_last, min_segment, end_buffer):
            continue
        model = fit_trend_seasonal(t, calendar_month, y, combo)
        if best is None or model["rss"] < best["rss"]:
            best = model
    if best is None:
        return None

    # Local refinement at one-month resolution.
    improved = True
    while improved:
        improved = False
        for i in range(n_breakpoints):
            for shift in (-1, 1):
                trial = list(best["breakpoints"])
                trial[i] += shift
                if not breakpoints_valid(trial, t_first, t_last, min_segment, end_buffer):
                    continue
                model = fit_trend_seasonal(t, calendar_month, y, trial)
                if model["rss"] < best["rss"] - 1e-9:
                    best = model
                    improved = True
    return best


def select_model(t, calendar_month, y, n_breakpoints="auto", max_breakpoints=3):
    """Fit the trend + seasonal model. With "auto", choose 0-3 breakpoints by BIC.

    Starting from a straight-line trend, a fit with more breakpoints is accepted only if its
    BIC is at least MIN_BIC_DROP lower than the fit currently chosen.

    Returns (model, table) where table lists the BIC for each number of breakpoints tried.
    """
    for m in range(1, 13):
        if np.sum(calendar_month == m) < 2:
            raise ValueError(f"Calendar month {m} has fewer than 2 observations; "
                             "the record is too short to estimate the seasonal cycle.")
    if n_breakpoints == "auto":
        options = range(0, max_breakpoints + 1)
    else:
        options = [int(n_breakpoints)]

    fits = {}
    for n in options:
        model = best_breakpoints(t, calendar_month, y, n)
        if model is not None:
            fits[n] = model
    if not fits:
        raise ValueError("Record too short for the requested number of breakpoints.")

    chosen = min(fits)
    for n in sorted(fits):
        if fits[n]["bic"] <= fits[chosen]["bic"] - MIN_BIC_DROP:
            chosen = n
    table = pd.DataFrame({
        "breakpoints": list(fits),
        "BIC": [round(fits[n]["bic"], 1) for n in fits],
        "RMSE (cm)": [round(np.sqrt(fits[n]["rss"] / fits[n]["n"]), 2) for n in fits],
    }).set_index("breakpoints")
    return fits[chosen], table


def evaluate_model(model, t, calendar_month):
    """Return (trend, seasonal) arrays at times t.

    The seasonal values are shifted to average zero over the 12 months and the trend is
    shifted up by the same amount, so trend + seasonal is unchanged. Because the monthly
    columns are fitted jointly with the trend, each seasonal value equals the mean of the
    detrended observations for that calendar month.
    """
    n_trend = 1 + len(model["breakpoints"])
    trend_coef = model["coef"][:n_trend]
    monthly_level = model["coef"][n_trend:]
    offset = monthly_level.mean()

    trend_cols = [t] + [np.maximum(t - b, 0) for b in model["breakpoints"]]
    trend = np.column_stack(trend_cols).astype(float) @ trend_coef + offset
    seasonal = monthly_level[np.asarray(calendar_month) - 1] - offset
    return trend, seasonal


def segment_slopes(model, start_date, end_date):
    """Table of trend segments with their slopes in cm/yr."""
    n_trend = 1 + len(model["breakpoints"])
    coef = model["coef"][:n_trend]
    edges = [start_date] + [start_date + pd.DateOffset(months=int(b)) for b in model["breakpoints"]] + [end_date]
    rows = []
    for i in range(n_trend):
        slope_per_month = coef[: i + 1].sum()  # each hinge adds its change to the slope
        rows.append({"from": edges[i].strftime("%Y-%m"),
                     "to": edges[i + 1].strftime("%Y-%m"),
                     "slope (cm/yr)": round(12 * slope_per_month, 2)})
    return pd.DataFrame(rows)


def fill_gaps(df, n_breakpoints="auto"):
    """Fill missing months. Observed values are kept as they are.

    filled = trend + seasonal + residual correction, where the residual (observed minus
    trend + seasonal) is interpolated linearly across each gap from the observed months on
    either side. A one-month gap then behaves like interpolation between its neighbours,
    and a long gap keeps the seasonal shape while joining the observations at both ends.

    Returns (result, model, bic_table). result has columns observed, upper, lower, trend,
    seasonal, correction, filled and is_filled.
    """
    start = df.index[0]
    t = months_since(df.index, start)
    calendar_month = np.asarray(df.index.month)
    has_data = df["observed"].notna().to_numpy()

    model, bic_table = select_model(t[has_data], calendar_month[has_data],
                                    df["observed"].to_numpy()[has_data], n_breakpoints)
    trend, seasonal = evaluate_model(model, t, calendar_month)

    result = df.copy()
    result["trend"] = trend
    result["seasonal"] = seasonal
    residual = result["observed"] - (trend + seasonal)  # NaN in the gaps
    # Linear across each gap; held constant past the first or last observation.
    result["correction"] = residual.interpolate(limit_direction="both").where(~has_data)
    result["is_filled"] = ~has_data
    result["filled"] = result["observed"].where(
        has_data, result["trend"] + result["seasonal"] + result["correction"])
    return result, model, bic_table


def rmse(predicted, truth):
    """Root-mean-square error, ignoring months where the truth is missing."""
    error = (np.asarray(predicted) - np.asarray(truth))[~np.isnan(truth)]
    return float(np.sqrt(np.mean(error ** 2)))


def hide_and_fill(df, hidden_rows, n_breakpoints):
    """Hide the given rows, refill them three ways, and return (truth, predictions dict)."""
    test = df.copy()
    test.iloc[hidden_rows, test.columns.get_loc("observed")] = np.nan
    test_result, _, _ = fill_gaps(test, n_breakpoints)
    hidden = test_result.iloc[hidden_rows]
    predictions = {
        "Trend + seasonal only": (hidden["trend"] + hidden["seasonal"]).to_numpy(),
        "Linear interpolation": (test["observed"].interpolate(limit_direction="both")
                                 .iloc[hidden_rows].to_numpy()),
        "Hybrid (used here)": hidden["filled"].to_numpy(),
    }
    return df["observed"].iloc[hidden_rows].to_numpy(), predictions


def fill_skill_check(df, n_breakpoints="auto", fraction=0.10, seed=42,
                     block_months=11, n_blocks=6):
    """Compare three ways of filling months that are actually observed.

    Test 1 hides a random `fraction` of observed months (fixed seed), one at a time.
    Test 2 hides `n_blocks` separate runs of `block_months` consecutive months, spread
    through the record, to mimic the GRACE/GRACE-FO gap. Returns a table of RMSE in cm.
    """
    rng = np.random.default_rng(seed)
    observed_rows = np.flatnonzero(df["observed"].notna().to_numpy())
    n_hide = max(1, int(round(fraction * len(observed_rows))))
    hidden_rows = np.sort(rng.choice(observed_rows, size=n_hide, replace=False))
    truth, predictions = hide_and_fill(df, hidden_rows, n_breakpoints)
    single = {name: rmse(values, truth) for name, values in predictions.items()}

    # Blocks start at evenly spaced positions, at least a year from either end.
    first_starts = np.linspace(12, len(df) - 12 - block_months, n_blocks).astype(int)
    block_truth, block_predictions = [], {name: [] for name in single}
    for first in first_starts:
        truth, predictions = hide_and_fill(df, np.arange(first, first + block_months),
                                           n_breakpoints)
        block_truth.append(truth)
        for name, values in predictions.items():
            block_predictions[name].append(values)
    block_truth = np.concatenate(block_truth)
    blocks = {name: rmse(np.concatenate(values), block_truth)
              for name, values in block_predictions.items()}

    return pd.DataFrame({
        f"{n_hide} single months (RMSE, cm)": single,
        f"{n_blocks} blocks of {block_months} months (RMSE, cm)": blocks,
    }).round(2)


# ----------------------------------------------------------------------------
# Recharge: water table fluctuation (WTF) method
# ----------------------------------------------------------------------------

def auto_water_year_start(result):
    """Calendar month with the lowest seasonal value (the climatological trough)."""
    by_month = result.groupby(result.index.month)["seasonal"].mean()
    return int(by_month.idxmin())


def water_year_label(first_month_date, start_month):
    """Name a water year by the calendar year in which it starts."""
    return first_month_date.year


def list_water_years(index, start_month):
    """List (label, first_month, last_month) for every complete water year in the index."""
    starts = index[index.month == start_month]
    years = []
    for first in starts:
        last = first + pd.DateOffset(months=11)
        if last <= index[-1]:
            years.append((water_year_label(first, start_month), first, last))
    return years


def parse_overrides(overrides):
    """Convert {year: {"peak": "YYYY-MM", "trough": "YYYY-MM"}} to Timestamps."""
    parsed = {}
    for year, picks in (overrides or {}).items():
        unknown = set(picks) - {"peak", "trough"}
        if unknown:
            raise ValueError(f"OVERRIDES[{year}] has unknown keys {unknown}; use 'peak' and/or 'trough'.")
        parsed[int(year)] = {key: pd.Timestamp(value).to_period("M").to_timestamp()
                             for key, value in picks.items()}
    return parsed


def recession_line(series, start, end):
    """Least-squares line through series[start:end].

    Returns (slope in cm/month, intercept at `start`) or None if fewer than 2 months.
    """
    window = series.loc[start:end]
    if len(window) < 2:
        return None
    x = months_since(window.index, start)
    slope, intercept = np.polyfit(x, window.to_numpy(), 1)
    return slope, intercept


def water_table_fluctuation(result, start_month, overrides=None):
    """Pick S_P, S_B and S_L for every complete water year and compute recharge (cm/yr).

    Peak and trough dates are picked on the detrended series (filled - trend) so that a
    long-term rise or decline can't push them to the edge of the water year. The S values
    and the recession line use the filled series itself.

    S_P  value at the highest detrended point in the water year (wet-season peak)
    S_B  value at the lowest detrended point between the previous peak and this peak
    S_A  previous year's peak, where the recession starts
    S_L  recession line (fitted from S_A to S_B, or to the MIN_RECESSION_MONTHS months
         ending at S_B if that is longer) extended to the date of S_P
    R_S = S_P - S_B          R_D = max(S_B - S_L, 0)
    R1  = R_S (lower)        R2  = R_S + R_D (upper)
    """
    series = result["filled"]
    detrended = result["filled"] - result["trend"]
    is_filled = result["is_filled"]
    overrides = parse_overrides(overrides)
    years = list_water_years(series.index, start_month)
    labels = [label for label, _, _ in years]
    for year in overrides:
        if year not in labels:
            raise ValueError(f"OVERRIDES year {year} is not a complete water year in the data "
                             f"({labels[0]}-{labels[-1]}).")

    rows = []
    previous_peak = None
    for label, first, last in years:
        picks = overrides.get(label, {})

        # Peak: the highest detrended value in this water year, unless overridden.
        peak_date = picks.get("peak", detrended.loc[first:last].idxmax())

        # Recession starts at the previous peak (or the start of the record).
        recession_start = previous_peak if previous_peak is not None else series.index[0]
        if peak_date <= recession_start:
            raise ValueError(f"Water year {label}: peak {peak_date:%Y-%m} is not after the "
                             f"previous peak {recession_start:%Y-%m}. Check OVERRIDES.")

        # Trough: the lowest detrended value between the previous peak and this peak.
        if "trough" in picks:
            trough_date = picks["trough"]
        else:
            trough_date = detrended.loc[recession_start:peak_date - ONE_MONTH].idxmin()
        if not recession_start <= trough_date < peak_date:
            raise ValueError(f"Water year {label}: trough {trough_date:%Y-%m} must fall between "
                             f"{recession_start:%Y-%m} and the peak {peak_date:%Y-%m}.")

        s_p = series[peak_date]
        s_b = series[trough_date]

        # Fit the recession line on at least MIN_RECESSION_MONTHS months ending at S_B.
        fit_start = recession_start
        shortest_start = trough_date - pd.DateOffset(months=MIN_RECESSION_MONTHS - 1)
        if fit_start > shortest_start:
            fit_start = max(shortest_start, series.index[0])

        # Extend the recession line to the peak date. A flat or rising "recession" means
        # there is no drainage to correct for, so S_L = S_B.
        line = recession_line(series, fit_start, trough_date)
        if line is None or line[0] >= 0:
            slope = np.nan if line is None else line[0]
            s_l = s_b
        else:
            slope, intercept = line
            s_l = intercept + slope * months_since([peak_date], fit_start)[0]

        # Flag lines extended more than twice as far as the span they were fitted on.
        fitted_span = months_since([trough_date], fit_start)[0]
        extrapolated = months_since([peak_date], trough_date)[0]
        long_extrapolation = bool(extrapolated > 2 * fitted_span)

        r_s = s_p - s_b
        r_d = max(s_b - s_l, 0.0)

        # Which parts of the pick rely on filled (estimated) months?
        parts = []
        if is_filled[peak_date]:
            parts.append("peak")
        if is_filled[trough_date]:
            parts.append("trough")
        if is_filled.loc[fit_start:trough_date].any():
            parts.append("recession")

        rows.append({
            "water_year": label,
            "recession_start": recession_start,
            "fit_start": fit_start,
            "trough_date": trough_date,
            "peak_date": peak_date,
            "S_P": s_p, "S_B": s_b, "S_L": s_l,
            "recession_slope": 12 * slope,  # cm/yr
            "R_S": r_s, "R_D": r_d,
            "R1": r_s, "R2": r_s + r_d,
            "long_extrapolation": long_extrapolation,
            "uses_filled_data": bool(parts),
            "filled_parts": ", ".join(parts),
            "overridden": ", ".join(sorted(picks)),
        })
        previous_peak = peak_date

    return pd.DataFrame(rows).set_index("water_year")


def summary_table(wtf, extra_columns=()):
    """Rounded per-year table with a Mean row at the bottom, for display and export."""
    columns = ["S_P", "S_B", "S_L", "R_S", "R_D", "R1", "R2"] + list(extra_columns)
    table = wtf[columns].round(2)
    table.insert(0, "peak", wtf["peak_date"].dt.strftime("%Y-%m"))
    table.insert(0, "trough", wtf["trough_date"].dt.strftime("%Y-%m"))
    table["long_extrapolation"] = wtf["long_extrapolation"]
    table["uses_filled_data"] = wtf["uses_filled_data"]
    table["filled_parts"] = wtf["filled_parts"]
    table.index = table.index.astype(str)
    mean_row = {col: round(wtf[col].mean(), 2) for col in columns}
    mean_row.update({"trough": "", "peak": "", "long_extrapolation": "", "uses_filled_data": "",
                     "filled_parts": ""})
    table.loc["Mean"] = pd.Series(mean_row)
    return table


# ----------------------------------------------------------------------------
# Plots
# ----------------------------------------------------------------------------

def shade_long_gaps(ax, result, min_months=6):
    """Shade runs of >= min_months missing months and label them."""
    for first, last in missing_runs(result["is_filled"]):
        n_months = months_since([last], first)[0] + 1
        if n_months < min_months:
            continue
        ax.axvspan(first - pd.DateOffset(days=15), last + pd.DateOffset(days=15),
                   color=COLORS["gray"], alpha=0.12, lw=0, zorder=0)
        is_mission_gap = first <= pd.Timestamp("2018-05-01") and last >= pd.Timestamp("2017-07-01")
        text = "GRACE / GRACE-FO gap" if is_mission_gap else f"{n_months}-month gap"
        middle = first + (last - first) / 2
        ax.text(middle, 0.98, text, transform=ax.get_xaxis_transform(), ha="center",
                va="top", fontsize=8.5, color=COLORS["gray"])


def plot_raw(ax, df, variable):
    """Observed series with the +/- 1 sigma band. NaN months show as breaks."""
    ax.fill_between(df.index, df["lower"], df["upper"], color=COLORS["blue"], alpha=0.15,
                    lw=0, label="±1σ")
    ax.plot(df.index, df["observed"], color=COLORS["observed"], lw=1.2, marker="o", ms=2.5,
            label=f"Observed {variable}")
    ax.set_ylabel(f"{variable} (cm)")
    ax.legend(loc="best")


def plot_gap_fill(ax, result, model=None, variable="GWSa"):
    """Observed series, fitted trend, and filled months highlighted.

    If the fitted model is given, each trend segment is labelled with its slope.
    """
    shade_long_gaps(ax, result)
    # A thin orange line through the filled months and their observed neighbours.
    near_fill = (result["is_filled"]
                 | result["is_filled"].shift(1, fill_value=False)
                 | result["is_filled"].shift(-1, fill_value=False))
    ax.plot(result.index, result["filled"].where(near_fill), color=COLORS["filled"], lw=1,
            alpha=0.8)
    ax.plot(result.index, result["observed"], color=COLORS["observed"], lw=1.2, marker="o",
            ms=2.5, label="Observed")
    ax.plot(result.index, result["trend"], color=COLORS["trend"], lw=2, label="Fitted trend")
    filled = result[result["is_filled"]]
    ax.plot(filled.index, filled["filled"], ls="none", marker="o", ms=4.5,
            color=COLORS["filled"], label="Filled (trend + seasonal + residual)")
    if model is not None:
        segments = segment_slopes(model, result.index[0], result.index[-1])
        lowest_label = np.inf
        for _, segment in segments.iterrows():
            seg_start, seg_end = pd.Timestamp(segment["from"]), pd.Timestamp(segment["to"])
            middle = (seg_start + (seg_end - seg_start) / 2).to_period("M").to_timestamp()
            # Place the label just under the lowest point within a year of the middle.
            nearby = result["filled"].loc[middle - pd.DateOffset(months=12):
                                          middle + pd.DateOffset(months=12)]
            ax.annotate(f"{segment['slope (cm/yr)']:+.1f} cm/yr", (middle, nearby.min()),
                        xytext=(0, -6), textcoords="offset points", ha="center", va="top",
                        fontsize=8.5, color=COLORS["gray"])
            lowest_label = min(lowest_label, nearby.min())
        # Leave room under the lowest label.
        bottom, top = ax.get_ylim()
        ax.set_ylim(min(bottom, lowest_label - 0.1 * (top - bottom)), top)
    ax.set_ylabel(f"{variable} (cm)")
    ax.legend(loc="best")


def plot_series_with_picks(ax, result, wtf, variable="GWSa"):
    """Filled series with the picked peaks and troughs marked."""
    ax.plot(result.index, result["filled"], color=COLORS["observed"], lw=1.1)
    filled = result[result["is_filled"]]
    ax.plot(filled.index, filled["filled"], ls="none", marker="o", ms=3.5,
            color=COLORS["filled"], label="Filled month")
    ax.plot(wtf["peak_date"], wtf["S_P"], ls="none", marker="^", ms=7, color=COLORS["blue"],
            label="Peak $S_P$")
    ax.plot(wtf["trough_date"], wtf["S_B"], ls="none", marker="v", ms=7,
            color=COLORS["recession"], label="Trough $S_B$")
    ax.set_ylabel(f"{variable} (cm)")
    ax.legend(loc="best", ncol=3)


def draw_bracket(ax, x, y_low, y_high, color):
    """Vertical bracket from y_low to y_high at date x."""
    tick = pd.DateOffset(days=10)
    ax.plot([x, x], [y_low, y_high], color=color, lw=1.6)
    ax.plot([x - tick, x + tick], [y_low, y_low], color=color, lw=1.6)
    ax.plot([x - tick, x + tick], [y_high, y_high], color=color, lw=1.6)


def plot_wtf_year(ax, result, row, months_before=2, months_after=4):
    """One water year: series, recession line, S_P/S_B/S_L and the R_S/R_D brackets.

    The series is shown from months_before the recession start to months_after the peak.
    """
    start = min(row["recession_start"], row["fit_start"]) - pd.DateOffset(months=months_before)
    end = row["peak_date"] + pd.DateOffset(months=months_after)
    window = result.loc[start:end]
    ax.plot(window.index, window["filled"], color=COLORS["observed"], lw=1.2, marker="o",
            ms=3, zorder=3)
    filled = window[window["is_filled"]]
    ax.plot(filled.index, filled["filled"], ls="none", marker="o", ms=4.5,
            color=COLORS["filled"], zorder=4)

    # Recession line from S_A to the peak date (only drawn when it slopes down).
    if row["recession_slope"] < 0:
        n_months = months_since([row["peak_date"]], row["fit_start"])[0]
        y_start = row["S_L"] - row["recession_slope"] / 12 * n_months
        ax.plot([row["fit_start"], row["peak_date"]], [y_start, row["S_L"]],
                color=COLORS["recession"], ls="--", lw=1.4, zorder=2)
    # Level of S_B carried forward to the peak date.
    ax.plot([row["trough_date"], row["peak_date"]], [row["S_B"], row["S_B"]],
            color=COLORS["gray"], ls=":", lw=1.2, zorder=2)

    bracket_x = row["peak_date"] + pd.DateOffset(days=25)
    draw_bracket(ax, bracket_x, row["S_B"], row["S_P"], COLORS["blue"])
    if row["R_D"] > 0:
        draw_bracket(ax, bracket_x, row["S_L"], row["S_B"], COLORS["recession"])

    ax.plot(row["peak_date"], row["S_P"], "o", ms=7, mfc="white", mec=COLORS["blue"], mew=1.8,
            zorder=5)
    ax.plot(row["trough_date"], row["S_B"], "o", ms=7, mfc="white", mec=COLORS["blue"],
            mew=1.8, zorder=5)
    if row["R_D"] > 0:
        ax.plot(row["peak_date"], row["S_L"], "o", ms=7, mfc="white",
                mec=COLORS["recession"], mew=1.8, zorder=5)


def plot_wtf_grid(result, wtf, variable="GWSa", ncols=4):
    """Small multiples: one panel per water year."""
    n = len(wtf)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 2.4 * nrows), squeeze=False)
    for ax, (year, row) in zip(axes.flat, wtf.iterrows()):
        plot_wtf_year(ax, result, row)
        flag = " *" if row["uses_filled_data"] else ""
        ax.set_title(f"{year}{flag}   R1 {row['R1']:.1f}, R2 {row['R2']:.1f}", fontsize=9,
                     loc="left")
        ax.tick_params(labelsize=7.5)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
        ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
    for ax in axes.flat[n:]:
        ax.set_visible(False)
    for ax in axes[:, 0]:
        ax.set_ylabel(f"{variable} (cm)", fontsize=8.5)
    fig.text(0.01, 0.002, "* uses filled months.  Blue bracket: $R_S$.  Red bracket: $R_D$.  "
             "Dashed red: recession line.", fontsize=8.5, color=COLORS["gray"])
    fig.tight_layout(rect=(0, 0.02, 1, 1))
    return fig


def plot_recharge_bars(ax, wtf):
    """R1 and R2 per water year with their means."""
    x = np.arange(len(wtf))
    width = 0.4
    ax.bar(x - width / 2, wtf["R1"], width, color=COLORS["observed"], label="R1 (method 1)")
    ax.bar(x + width / 2, wtf["R2"], width, color=COLORS["blue"], alpha=0.55,
           label="R2 (method 2)")
    ax.axhline(wtf["R1"].mean(), color=COLORS["observed"], ls="--", lw=1.2,
               label=f"Mean R1 = {wtf['R1'].mean():.2f}")
    ax.axhline(wtf["R2"].mean(), color=COLORS["blue"], ls="--", lw=1.2,
               label=f"Mean R2 = {wtf['R2'].mean():.2f}")
    names = [f"{year}*" if flag else str(year)
             for year, flag in zip(wtf.index, wtf["uses_filled_data"])]
    ax.set_xticks(x, names, rotation=90, fontsize=8)
    ax.set_ylabel("Recharge (cm/yr)")
    ax.set_xlabel("Water year (* uses filled months)")
    ax.set_ylim(0, 1.25 * wtf["R2"].max())
    ax.legend(loc="upper left", ncol=2)

The next cell finds the input file. If `CSV_FILE` is the default sample and isn't here, it is
downloaded. In Colab, any other file that isn't found opens an upload dialog; locally, put the
CSV next to the notebook or give its full path.

In [ ]:
SAMPLE_URL = ("https://raw.githubusercontent.com/Aquaveo/training.geoglows.org/main/"
              "docs/static/files/grace/sample_iullemeden.csv")
IN_COLAB = "google.colab" in sys.modules

csv_path = Path(CSV_FILE)
if not csv_path.exists():
    if csv_path.name == Path(SAMPLE_URL).name:
        print(f"Downloading the sample file to {csv_path} ...")
        urllib.request.urlretrieve(SAMPLE_URL, csv_path)
    elif IN_COLAB:
        from google.colab import files
        print(f"{CSV_FILE} not found. Choose the CSV exported from the web app:")
        uploaded = files.upload()
        csv_path = Path(next(iter(uploaded)))
    else:
        raise FileNotFoundError(f"{CSV_FILE} not found. Put it next to this notebook "
                                "or set CSV_FILE to its full path.")
print("Input file:", csv_path)
set_plot_style()

## 2. Load and inspect

The series is placed on a complete monthly calendar. Months that are blank in the file (or
absent from it) are treated as missing. Most are single months skipped for battery
management late in the GRACE mission; the longest is the 11-month gap between the GRACE and
GRACE-FO missions (July 2017 – May 2018).

In [ ]:
data = load_grace_csv(csv_path, VARIABLE)
missing = data.index[data["observed"].isna()]

print(f"{VARIABLE}: {data.index[0]:%Y-%m} to {data.index[-1]:%Y-%m}, {len(data)} months")
print(f"Missing months: {len(missing)}")
print(", ".join(missing.strftime("%Y-%m")))

fig, ax = plt.subplots()
plot_raw(ax, data, VARIABLE)
ax.set_title(f"{VARIABLE} from {csv_path.name}", loc="left")
plt.show()

## 3. Fill the gaps

Each month is modeled as **trend + seasonal**:

- **Trend**: a continuous line whose slope can change at up to `N_BREAKPOINTS` months. Each
  segment must span at least 36 months, and no breakpoint may fall within 48 months of either
  end of the record, where a single unusual year would have the most pull. With `"auto"`, the notebook fits 0, 1, 2 and 3 breakpoints and compares them
  by BIC (Bayesian Information Criterion), which rewards a better fit but penalizes every
  extra breakpoint. A fit with more breakpoints is used only if it lowers the BIC by at
  least 10, so a single wet or dry year can't create a slope change on its own.
- **Seasonal**: for each calendar month, the average of the observed values after the trend
  is removed.

A missing month is filled with trend + seasonal plus a **residual correction**: the
difference between the observations and trend + seasonal, interpolated linearly across the
gap from the observed months on either side. A single missing month then sits between its
neighbours, and the 11-month GRACE/GRACE-FO gap keeps the seasonal shape while joining the
observations at both ends. Observed months are never changed.

In [ ]:
result, model, bic_table = fill_gaps(data, N_BREAKPOINTS)

print("Fit for each number of breakpoints (lower BIC is better):")
display(bic_table)
print(f"Using {len(model['breakpoints'])} breakpoint(s).")
print("\nTrend segments:")
display(segment_slopes(model, data.index[0], data.index[-1]))

fig, ax = plt.subplots()
plot_gap_fill(ax, result, model, VARIABLE)
ax.set_title(f"{VARIABLE}: observed and filled months", loc="left")
plt.show()

**How good are the filled values?** The cell below hides observed months, fills them again,
and compares the result with the real values. It does this twice: once for a random 10% of
months hidden one at a time (like the short gaps in the record), and once for six separate
11-month blocks (like the GRACE/GRACE-FO gap). Each test reports three fills: trend + seasonal
alone, straight-line interpolation, and the hybrid used in this notebook. Compare the errors
with the 1σ uncertainty of the observations printed below the table. This cell takes about a
minute.

In [ ]:
display(fill_skill_check(data, N_BREAKPOINTS))
print(f"Typical 1-sigma uncertainty of the observations: "
      f"{(data['upper'] - data['observed']).median():.2f} cm")

## 4. Recharge by the water table fluctuation method

The WTF method treats each year's rise in storage as recharge. It works one **water year** at
a time. With `WATER_YEAR_START_MONTH = "auto"` the water year starts in the calendar month
where the seasonal component is lowest, so each water year runs from one dry-season low to the
next and contains one wet-season rise. A water year is named by the calendar year it starts in.

For each water year:

- **S_P**: the wet-season peak, the highest point in the water year.
- **S_B**: the trough, the lowest point between the previous year's peak and this peak.
- **Recession line**: a straight line fitted to the values from the previous peak (S_A) down
  to S_B (at least 4 months; if S_A and S_B are closer, the 4 months ending at S_B), extended
  to the date of S_P. Its value there is **S_L**, where storage would have
  been had the decline continued without recharge. If that line is flat or rising, S_L = S_B.
- **R_S** = S_P − S_B and **R_D** = S_B − S_L.
- **Method 1**: R1 = R_S (lower estimate). **Method 2**: R2 = R_S + R_D = S_P − S_L (upper
  estimate). Both are in cm/yr of water.

The peak and trough months are found after removing the long-term trend, so a steady
decline (from pumping, say) can't push the peak to the first month of the water year. The
S values themselves are read from the filled series.

Years whose peak, trough or recession line uses filled months are flagged in the table.

**Two cautions.**

- R2 extends the recession line well beyond the months it was fitted on, so treat it as an
  upper bound. Years where the line is extended more than twice as far as the span it was
  fitted on are marked `long_extrapolation` in the table; their R2 is the least reliable. Where storage has a strong long-term decline from pumping,
  the recession line includes that decline, and R_D then counts it as recharge.
- The 1σ uncertainty of GRACE storage (about 2 cm for the Niger sample) is the same size as a
  typical R_S, so a single year's value is uncertain. Means over several years are more robust.

In [ ]:
seasonal_by_month = result.groupby(result.index.month)["seasonal"].mean()
if WATER_YEAR_START_MONTH == "auto":
    start_month = auto_water_year_start(result)
    print(f"Water year starts in month {start_month} (lowest seasonal value).")
else:
    start_month = int(WATER_YEAR_START_MONTH)
    print(f"Water year starts in month {start_month} (set by WATER_YEAR_START_MONTH).")

fig, ax = plt.subplots(figsize=(6, 2.6))
bar_colors = [COLORS["filled"] if m == start_month else COLORS["gray"] for m in range(1, 13)]
ax.bar(seasonal_by_month.index, seasonal_by_month.values, color=bar_colors)
ax.set_xticks(range(1, 13), ["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"])
ax.set_ylabel("Seasonal (cm)")
ax.set_title("Average seasonal cycle (orange = water year start)", loc="left", fontsize=10)
plt.show()

wtf = water_table_fluctuation(result, start_month, OVERRIDES)
display(summary_table(wtf))

In [ ]:
fig, ax = plt.subplots()
plot_series_with_picks(ax, result, wtf, VARIABLE)
ax.set_title("Picked peaks and troughs", loc="left")
plt.show()

Check each year below. If a peak or trough was picked in the wrong place (for example, a
noisy spike), add the correct month to `OVERRIDES` in the settings cell, e.g.
`OVERRIDES = {2010: {"peak": "2010-09", "trough": "2010-05"}}`, and run all cells again.

In [ ]:
fig = plot_wtf_grid(result, wtf, VARIABLE)
plt.show()

In [ ]:
fig, ax = plt.subplots()
plot_recharge_bars(ax, wtf)
ax.set_title("Annual recharge estimates", loc="left")
plt.show()

## 5. Convert to volume (optional)

Set `AREA_KM2` to the area of your region in km² to express recharge as a volume.
1 cm over 1 km² is 10,000 m³.

In [ ]:
AREA_KM2 = None  # e.g. 500000

volume_columns = []
if AREA_KM2:
    for method in ("R1", "R2"):
        wtf[f"{method}_km3"] = wtf[method] / 1e5 * AREA_KM2         # cm -> km, times km2
        wtf[f"{method}_Mm3"] = wtf[f"{method}_km3"] * 1000          # million m3
        volume_columns += [f"{method}_km3", f"{method}_Mm3"]
    display(summary_table(wtf, volume_columns))
else:
    print("AREA_KM2 not set; skipping volume conversion.")

## 6. Export

Two CSV files are written next to the input file name:

- `<name>_filled.csv`: Date, observed, filled, is_filled, trend, seasonal, correction
  (the residual correction, filled months only)
- `<name>_recharge.csv`: the per-year table above, with means in the last row

In Colab they are also downloaded to your computer.

In [ ]:
filled_path = Path(f"{csv_path.stem}_filled.csv")
recharge_path = Path(f"{csv_path.stem}_recharge.csv")

export = result[["observed", "filled", "is_filled", "trend", "seasonal", "correction"]].round(3)
export.index = export.index.strftime("%Y-%m-%d")
export.index.name = "Date"
export.to_csv(filled_path)

summary_table(wtf, volume_columns).to_csv(recharge_path, index_label="water_year")
print("Wrote", filled_path, "and", recharge_path)

if IN_COLAB:
    from google.colab import files
    files.download(str(filled_path))
    files.download(str(recharge_path))